# 05. Lecke - Ügynöki RAG


## Beállítás

Ez a jegyzetfüzet a Microsoft Agent Framework használatával mutatja be az Agentic RAG (Retrieval-Augmented Generation) mintát.

**Előfeltételek:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — az Azure AI Search szolgáltatás végpontja
- `AZURE_SEARCH_API_KEY` — az Azure AI Search API kulcsa
- Környezeti változókon keresztül konfigurált Azure OpenAI telepítés
- Azure CLI hitelesítve (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi az Agentic RAG?

A hagyományos RAG egy rögzített folyamatot követ: dokumentumokat kér le, majd választ generál. Az **Agentic RAG** ennél továbbmegy azzal, hogy az ügynök autonómiát kap annak eldöntésére, **mikor** és **hogyan** kérdezze le az információkat.

Az Agentic RAG esetében az ügynök képes:
- **Dönteni** arról, szükséges-e lekérés a kérdés megválaszolása előtt
- **Kiválasztani**, mely adatforráshoz vagy eszközhöz forduljon
- **Kiértékelni** a lekért eredményeket, és ha az első próbálkozás nem elég, további lekéréseket végezni
- **Összegezni** a több lekérési lépésből származó információkat egy koherens válaszban

Ez azáltal teszi az ügynököt rugalmasabbá és pontosabbá a statikus lekérés- majd generálás folyamatához képest.


## Keresőeszköz létrehozása

Az Agentic RAG-ben a külső adatforrások **eszközökként** vannak becsomagolva, amelyeket az ügynök igény szerint hívhat meg. Ez lehetővé teszi, hogy az ügynök a lekérést csak egy másik műveletként kezelje, nem pedig kötelező lépésként.

Az alábbiakban egy utazási tudásbázist definiálunk, és eszközként tesszük elérhetővé az ügynök számára, hogy célállomás-információkat kereshessen benne.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## A RAG ügynök létrehozása

Most létrehozunk egy olyan ügynököt, akit arra utasítanak, hogy **mindig információt szerezzen be a válaszadás előtt**. Az ügynök a `search_travel_knowledge` eszközt használja, hogy válaszait a tudásbázison alapozza meg, ahelyett, hogy saját tanító adataira támaszkodna.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Ismétlődő lekérdezés — A Maker-Checker minta

Az Agentic RAG egyik fő előnye az **ismétlődő lekérdezés**. Az ügynök több keresési kört is végezhet, hogy ellenőrizze, finomítsa vagy bővítse kezdeti eredményeit — hasonlóan a "maker-checker" munkafolyamathoz:

1. **Maker lépés**: Az ügynök lekéri a kezdeti információkat és megfogalmaz egy választ.
2. **Checker lépés**: Az ügynök további lekérdezéseket végez a részletek ellenőrzése vagy a hiányosságok pótlása érdekében.

Az alábbiban az ügynök egy olyan kérdést kap, amely több célpont összehasonlítását igényli, ez több keresésre ösztönzi.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Összefoglalás

Ebben a leckében megtanultad, hogyan építs fel egy **Agentic RAG** rendszert a Microsoft Agent Framework használatával:

- Az **Agentic RAG** lehetővé teszi az ügynökök számára, hogy autonóm módon döntsék el, mikor kérjenek le információt, így a lekérés dinamikus lesz, nem pedig rögzített.
- **Eszközök adatforrásként**: Külső tudásbázisok (például az Azure AI Search) eszközként vannak becsomagolva, amelyeket az ügynök meghívhat.
- **Ismétlődő lekérés**: A maker-checker minta lehetővé teszi, hogy az ügynök több lekérési kört hajtson végre — keresés, ellenőrzés és finomítás — mielőtt végleges választ ad.

Éles környezetben a memóriában lévő `TRAVEL_KNOWLEDGE_BASE` helyett egy valódi Azure AI Search indexet használnál, hogy nagy volumenű utazási dokumentumok lekérését kezeld.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Nyilatkozat**:
Ez a dokumentum az AI fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvi dokumentum tekintendő hiteles forrásnak. Kritikus információk esetén szakmai, emberi fordítást javaslunk. Nem vállalunk felelősséget az ebből eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
